In [1]:
import geopandas as gpd
from pyproj import Transformer
from requests import get, post
import pandas as pd
from shapely.geometry import Polygon, Point, MultiPolygon
import numpy as np

In [2]:
zoning_map = get('https://mapsa.slocity.org/hosting/rest/services/sloGISlayers/gisLandManagementWM/MapServer/5/query?where=1%3D1&f=pjson').json()


zones = pd.json_normalize(zoning_map['features'])
zones = zones.explode(column='geometry.rings')


arcgis_to_lat_log = Transformer.from_crs("EPSG:3857", "EPSG:4326")

zones['polygon'] = zones['geometry.rings'].map(lambda x: [arcgis_to_lat_log.transform(point[0], point[1])[::-1] for point in x])

zones = gpd.GeoDataFrame(zones['attributes.generalZone'].rename('zone'), geometry=zones['polygon'].map(Polygon))

del zoning_map
zones

,zone,geometry
0,R-1,"POLYGON ((-120.62803 35.26558, -120.62803 35.2..."
1,C/OS,"POLYGON ((-120.69361 35.31137, -120.69368 35.3..."
2,R-1,"POLYGON ((-120.64883 35.29331, -120.64884 35.2..."
3,R-1,"POLYGON ((-120.64659 35.29492, -120.64658 35.2..."
4,R-1,"POLYGON ((-120.64769 35.29465, -120.64769 35.2..."
...,...,...
1320,R-1,"POLYGON ((-120.63247 35.25076, -120.63251 35.2..."
1321,PF,"POLYGON ((-120.63378 35.2519, -120.63386 35.25..."
1322,R-1,"POLYGON ((-120.63309 35.25242, -120.63318 35.2..."
1323,PF,"POLYGON ((-120.63377 35.25326, -120.63381 35.2..."


In [5]:
license_data = pd.read_csv('./licenses_with_locs.txt')
license_data.head()

,DBA,Business type,Bus address,"Bus City, State",Start date,Close date,Account #,Owner Name 1,License description,Rate type (STD),type,long,lat
0,!ROMP,Apparel/Accessories,714 HIGUERA ST,"SAN LUIS OBISPO, CA 93401-3513",2005-04-06,2019-05-03,16453,"English, Karen",NaN,Downtown Association,Downtown Association,-120.664277,35.279094
1,"""""""Angel Lynn""""""",Misc Business Services,3860 S HIGUERA ST SP A24,"SAN LUIS OBISPO, CA 93401-7460",2009-06-08,2012-01-01,107052,"Block, Fay","Face Painting, Balloon Sculpture, Card Tricks,...",General Service,Service,-120.673262,35.247583
2,"""""""Slack""""""",Rental- Residential,2045 SLACK ST,"SAN LUIS OBISPO, CA 93405-2107",2009-10-28,2017-04-20,107394,"Scheel Larsen, Anne",Residential Property Rental,Residential Rental,Rental,-120.650013,35.295909
3,@ BITES,Restaurant,195 N SANTA ROSA ST,"SAN LUIS OBISPO, CA 93405-1322",2021-04-05,NaN,117190,"Le, Phuong",Fast Food Restaurant,General Retailer,Retail,-120.669502,35.295490
4,@ Nails,Beauty Salon,1519 FROOM RANCH WAY STE A BLDG E,"SAN LUIS OBISPO, CA 93405-7211",2008-01-23,NaN,106088,"Nguyen, Kieutrinh",Nail Salon,General Service,Service,-120.687969,35.252327


In [26]:
def get_zone(point : Point):
    return None if point == Point(0,0) else zones['zone'][zones.distance(point).idxmin()]

get_zone(Point(license_data['longitude'][0], license_data['latitude'][0]))

'C-D'

In [28]:
license_data[['longitude','latitude']].apply(lambda x: Point(x),axis=1).map(get_zone)

0       C-D
1       R-2
2       R-1
3       C-S
4       R-1
       ... 
8760    C-D
8761    R-2
8762      M
8763    C-S
8764    C-R
Length: 8765, dtype: object

In [138]:
offset = 0

data = pd.DataFrame()
while True:
    offset = len(data)
    new_data = pd.json_normalize(get(f'https://mapsa.slocity.org/hosting/rest/services/sloGISlayers/gisLandManagementWM/MapServer/4/query?where=1%3D1&f=pjson&resultOffset={offset}').json()['features'])
    data  = pd.concat([data, new_data]).reset_index(drop=True)
    if new_data.empty:
        break

parcels = gpd.GeoDataFrame(geometry=data['geometry.rings'].map(lambda x: [arcgis_to_lat_log.transform(point[0], point[1])[::-1] for point in x[0]]).map(Polygon)) # Convert points to shape objects

parcels['APN'] = data['attributes.APN']

del new_data
del data

def get_parcel(point : Point):
    return None if point == Point(0,0) else parcels['APN'][parcels.distance(point).idxmin()]

parcels

,geometry,APN
0,"POLYGON ((-120.67176 35.28901, -120.67176 35.2...",001-012-006
1,"POLYGON ((-120.67206 35.28854, -120.67229 35.2...",001-012-008
2,"POLYGON ((-120.67206 35.28854, -120.67212 35.2...",001-012-009
3,"POLYGON ((-120.67255 35.29033, -120.67255 35.2...",001-012-012
4,"POLYGON ((-120.67234 35.29033, -120.67234 35.2...",001-012-013
...,...,...
17326,"POLYGON ((-120.64412 35.26909, -120.64428 35.2...",004-785-011
17327,"POLYGON ((-120.64347 35.26948, -120.64373 35.2...",004-785-001
17328,"POLYGON ((-120.66348 35.27232, -120.66348 35.2...",003-623-002
17329,"POLYGON ((-120.64737 35.29002, -120.64757 35.2...",001-251-034


In [152]:
get_parcel(Point(license_data['longitude'][0], license_data['latitude'][0]))

'002-423-013'

In [173]:
license_data['Close_date']

KeyError: 'Close_date'